# Train Dynamics, Then Plan Through Them

Fit an action-conditioned model, score candidate rollouts, and test model exploitation.

## Before You Start

Course 13. Entry course: 12. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** embodied-04_latents.

## What You Will Be Able to Do

- Roll out a learned dynamics model
- Choose plans by a declared goal cost
- Measure compounding errors and replan

## Prediction becomes a planning primitive

A state or latent dynamics model predicts z_next=f(z,a). Training examples must include actions aligned with the transition they caused. A video-only predictor can learn temporal regularities without identifying how interventions change the world. For planning, sample candidate action sequences, roll each through the model, score their predicted costs, and choose a sequence. The lab uses linear dynamics z_next=A@z+B@a with explicit column-vector convention.

Our cost sums squared goal distances at every predicted step plus action_penalty times squared action magnitudes. Other tasks use terminal-only costs, obstacles, discounting, or uncertainty penalties; they produce different plans. Return the full predicted rollout and each candidate cost, not only the winning index. Tie-breaking is first index so results are reproducible.

## The planner can discover the model's mistakes

Small one-step errors can compound over a long horizon. A planner may select actions outside the training distribution where the model is overoptimistic. Evaluate selected plans in the actual simulator, not only in the learned model. Compare in-distribution and out-of-distribution actions, and report both predicted and observed costs.

Model predictive control executes the first action or short prefix, observes the actual new state, and replans. It can reduce accumulated error but does not guarantee safety or correct dynamics. Hierarchical planning separates long-range subgoals from short-horizon control; subgoal feasibility and model uncertainty remain open checks. The experiment trains on a known linear environment and adds a controlled model-error case. PushT and real robot contact dynamics are deliberately not represented by this simple environment.

## Worked Example: A two-step plan

Scalar dynamics z_next=z+a, start=0, goal=2, action penalty=0.

1. **Plan A:** Actions [1,1] predict states [1,2], cost (1-2)^2+(2-2)^2=1.
2. **Plan B:** Actions [2,0] predict [2,2], cost 0.
3. **Constraints matter:** If each action is bounded by 1, Plan B is invalid despite its lower unconstrained cost.

State dynamics, cost definition, and action constraints jointly define the planning problem.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Explicit linear latent rollouts and goal costs."""
import numpy as np

def plan_candidates(initial, dynamics_a, dynamics_b, candidates, goal, action_penalty=0.0):
    """Return (first best index, costs [C], predicted states [C,H,D]).

z_next=A@z+B@action. initial/goal [D], A [D,D], B [D,U], candidates [C,H,U].
Valid finite shapes C,H>=1 and nonnegative penalty assumed. Cost sums goal
squared distance at EVERY predicted step plus penalty*sum(action^2).
Convert numerical arrays to floating point before rollout and cost arithmetic.
Nonfinite active float64 intermediates raise FloatingPointError before selection.
A zero penalty skips action-effort arithmetic. With a positive penalty,
action squares and their sum must be finite even if scaling by a tiny
penalty could make the mathematical effort representable.
No input mutation. This function does not impose action bounds."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

Predicted states are [1,2], goal=3, no action penalty.

- Sum of squared goal distances

<details><summary>Worked answers</summary>

- 5: (1-3)^2+(2-3)^2=4+1=5.

</details>

## Experiment

Run `experiments/embodied_world.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Report held-out one-step error, multi-step error, predicted plan cost, and simulator cost.
- Training a linear state model is a planning prerequisite, not a reproduction of pixel-based PushT results.

## Transfer Problems

1. Add action bounds and reject invalid candidates before scoring.
2. Compare open-loop and replanned control under a disturbance at fixed planning budget.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[LeWorldModel](https://arxiv.org/html/2603.19312v1). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Explicit linear latent rollouts and goal costs."""
import numpy as np

def plan_candidates(initial, dynamics_a, dynamics_b, candidates, goal, action_penalty=0.0):
    """Return (first best index, costs [C], predicted states [C,H,D]).

z_next=A@z+B@action. initial/goal [D], A [D,D], B [D,U], candidates [C,H,U].
Valid finite shapes C,H>=1 and nonnegative penalty assumed. Cost sums goal
squared distance at EVERY predicted step plus penalty*sum(action^2).
Convert numerical arrays to floating point before rollout and cost arithmetic.
Nonfinite active float64 intermediates raise FloatingPointError before selection.
A zero penalty skips action-effort arithmetic. With a positive penalty,
action squares and their sum must be finite even if scaling by a tiny
penalty could make the mathematical effort representable.
No input mutation. This function does not impose action bounds."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
candidates = np.array([[[1.],[1.]],[[2.],[0.]],[[0.],[2.]]])
index,costs,states = plan_candidates(np.array([0.]),np.ones((1,1)),np.ones((1,1)),candidates,np.array([2.]))
assert index == 1 and np.allclose(costs,[1,0,4])
assert np.allclose(states[:,:,0],[[1,2],[2,2],[0,2]])
index,costs,_ = plan_candidates(np.array([0.]),np.ones((1,1)),np.ones((1,1)),candidates,np.array([2.]),2)
assert index == 0 and np.allclose(costs,[5,8,12])
inputs = (np.zeros(1),np.zeros((1,1)),np.full((1,1),1e-100),np.array([[[1e200]],[[1e199]]]),np.zeros(1))
originals = [value.copy() for value in inputs]
for penalty in (0.,-0.):
    with np.errstate(over='raise',invalid='raise'):
        index,costs,states = plan_candidates(*inputs,action_penalty=penalty)
    assert index == 1 and np.allclose(costs,[1e200,1e198]) and np.allclose(states,[[[1e100]],[[1e99]]]), 'Zero penalty must skip overflowing action effort'
    assert all(np.array_equal(value,before) for value,before in zip(inputs,originals))
for penalty in (1.,1e-300):
    try:
        plan_candidates(*inputs,action_penalty=penalty)
    except FloatingPointError:
        pass
    else:
        raise AssertionError('Active action-square overflow must fail before candidate selection')
    assert all(np.array_equal(value,before) for value,before in zip(inputs,originals))
positive_inputs = (np.zeros(1),np.zeros((1,1)),np.full((1,1),.01),np.array([[[100.]],[[10.]]]),np.zeros(1))
originals = [value.copy() for value in positive_inputs]
index,costs,states = plan_candidates(*positive_inputs,action_penalty=.5)
assert index == 1 and np.allclose(costs,[5001.,50.01]) and np.allclose(states,[[[1.]],[[.1]]])
assert all(np.array_equal(value,before) for value,before in zip(positive_inputs,originals))
for overflowing in [
    lambda: plan_candidates(np.array([1e308]),np.array([[2.]]),np.zeros((1,1)),np.zeros((2,1,1)),np.zeros(1),1.),
    lambda: plan_candidates(np.zeros(1),np.zeros((1,1)),np.ones((1,1)),np.array([[[1e200]],[[1e199]]]),np.zeros(1)),
    lambda: plan_candidates(np.array([-1e308]),np.ones((1,1)),np.zeros((1,1)),np.zeros((2,1,1)),np.array([1e308])),
    lambda: plan_candidates(np.zeros(2),np.zeros((2,2)),np.full((2,1),1e154),np.ones((2,1,1)),np.zeros(2)),
    lambda: plan_candidates(np.zeros(1),np.zeros((1,1)),np.zeros((1,2)),np.full((2,1,2),1e154),np.zeros(1),1.),
    lambda: plan_candidates(np.zeros(1),np.zeros((1,1)),np.zeros((1,1)),np.full((2,1,1),1e100),np.zeros(1),1e200),
    lambda: plan_candidates(np.zeros(1),np.zeros((1,1)),np.ones((1,1)),np.full((2,1,1),1e154),np.zeros(1),1.),
]:
    try:
        overflowing()
    except FloatingPointError:
        pass
    else:
        raise AssertionError('Nonfinite active rollout, squared-distance, reduction, or objective intermediates must fail explicitly')
for dtype in (np.int16,np.uint8):
    initial = np.array([0],dtype=dtype)
    a,b = np.zeros((1,1),dtype=dtype),np.ones((1,1),dtype=dtype)
    actions = np.array([[[200]],[[10]]],dtype=dtype)
    goal = np.array([0],dtype=dtype)
    inputs = (initial,a,b,actions,goal)
    originals = [value.copy() for value in inputs]
    index,costs,states = plan_candidates(*inputs)
    assert index == 1 and np.array_equal(costs,[40000.,100.]), 'Integer action costs must select action 10 over 200'
    assert np.array_equal(states,[[[200.]],[[10.]]])
    assert costs.dtype.kind == 'f' and states.dtype.kind == 'f'
    _,goal_costs,_ = plan_candidates(initial,a,b,actions,np.array([200],dtype=dtype))
    assert np.array_equal(goal_costs,[0.,36100.]), 'Unsigned goal subtraction must not wrap'
    assert all(np.array_equal(value,before) for value,before in zip(inputs,originals))
    initial = np.array([200,200],dtype=dtype)
    a,b = np.full((2,2),200,dtype=dtype),np.full((2,2),200,dtype=dtype)
    actions = np.array([[[200,200],[200,200]],[[10,10],[10,10]]],dtype=dtype)
    goal = np.array([200,200],dtype=dtype)
    inputs = (initial,a,b,actions,goal)
    originals = [value.copy() for value in inputs]
    index,costs,states = plan_candidates(*inputs,action_penalty=2.)
    expected_states = [[[160000.,160000.],[64080000.,64080000.]],[[84000.,84000.],[33604000.,33604000.]]]
    expected_costs = [2*(159800**2+64079800**2)+2*4*200**2,2*(83800**2+33603800**2)+2*4*10**2]
    assert index == 1 and np.allclose(states,expected_states) and np.allclose(costs,expected_costs), 'Rollout matrix products and action penalties must not overflow integer dtypes'
    assert all(np.array_equal(value,before) for value,before in zip(inputs,originals))
for dtype in (np.int64,np.uint64):
    actions = np.array([[[10**9]]*24,[[5*10**8]]*24],dtype=dtype)
    index,costs,_ = plan_candidates(np.zeros(1,dtype=dtype),np.zeros((1,1),dtype=dtype),np.ones((1,1),dtype=dtype),actions,np.zeros(1,dtype=dtype),3.)
    assert index == 1 and np.allclose(costs,[9.6e19,2.4e19]), 'Cost reductions must not overflow signed or unsigned integer accumulators'
rng = np.random.default_rng(1)
a,b = rng.normal(size=(2,2)),rng.normal(size=(2,3));c=rng.normal(size=(4,3,3));z=np.array([1.,-1.]);goal=np.array([.5,.2])
_,costs,rollouts = plan_candidates(z,a,b,c,goal,.1)
for i in range(4):
    state=z.copy();cost=0
    for t in range(3):
        state=a@state+b@c[i,t];cost+=sum((state-goal)**2)+.1*sum(c[i,t]**2)
        assert np.allclose(state,rollouts[i,t])
    assert np.isclose(cost,costs[i])
print('World-model planning checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Explicit linear latent rollouts and goal costs."""
import numpy as np


def plan_candidates(initial, dynamics_a, dynamics_b, candidates, goal, action_penalty=0.):
    """Return (first best index, costs [C], predicted states [C,H,D]).

    z_next=A@z+B@action. initial/goal [D], A [D,D], B [D,U], candidates [C,H,U].
    Valid finite shapes C,H>=1 and nonnegative penalty assumed. Cost sums goal
    squared distance at EVERY predicted step plus penalty*sum(action^2).
    Convert numerical arrays to floating point before rollout and cost arithmetic.
    Nonfinite active float64 intermediates raise FloatingPointError before selection.
    A zero penalty skips action-effort arithmetic. With a positive penalty,
    action squares and their sum must be finite even if scaling by a tiny
    penalty could make the mathematical effort representable.
    No input mutation. This function does not impose action bounds.
    """
    initial,dynamics_a,dynamics_b,candidates,goal = [np.asarray(value,float) for value in (initial,dynamics_a,dynamics_b,candidates,goal)]
    states = np.repeat(initial[None,:],len(candidates),axis=0)
    history = []
    with np.errstate(over='raise',invalid='raise'):
        for step in range(candidates.shape[1]):
            states = states@dynamics_a.T+candidates[:,step]@dynamics_b.T
            if not np.isfinite(states).all():
                raise FloatingPointError('nonfinite latent rollout')
            history.append(states.copy())
        rollout = np.stack(history,axis=1)
        costs = ((rollout-goal)**2).sum(axis=(1,2))
        if action_penalty != 0:
            costs = costs+action_penalty*(candidates**2).sum(axis=(1,2))
    if not np.isfinite(costs).all():
        raise FloatingPointError('nonfinite planning objective')
    return int(np.argmin(costs)),costs,rollout


In [ ]:
import numpy as np
candidates = np.array([[[1.],[1.]],[[2.],[0.]],[[0.],[2.]]])
index,costs,states = plan_candidates(np.array([0.]),np.ones((1,1)),np.ones((1,1)),candidates,np.array([2.]))
assert index == 1 and np.allclose(costs,[1,0,4])
assert np.allclose(states[:,:,0],[[1,2],[2,2],[0,2]])
index,costs,_ = plan_candidates(np.array([0.]),np.ones((1,1)),np.ones((1,1)),candidates,np.array([2.]),2)
assert index == 0 and np.allclose(costs,[5,8,12])
inputs = (np.zeros(1),np.zeros((1,1)),np.full((1,1),1e-100),np.array([[[1e200]],[[1e199]]]),np.zeros(1))
originals = [value.copy() for value in inputs]
for penalty in (0.,-0.):
    with np.errstate(over='raise',invalid='raise'):
        index,costs,states = plan_candidates(*inputs,action_penalty=penalty)
    assert index == 1 and np.allclose(costs,[1e200,1e198]) and np.allclose(states,[[[1e100]],[[1e99]]]), 'Zero penalty must skip overflowing action effort'
    assert all(np.array_equal(value,before) for value,before in zip(inputs,originals))
for penalty in (1.,1e-300):
    try:
        plan_candidates(*inputs,action_penalty=penalty)
    except FloatingPointError:
        pass
    else:
        raise AssertionError('Active action-square overflow must fail before candidate selection')
    assert all(np.array_equal(value,before) for value,before in zip(inputs,originals))
positive_inputs = (np.zeros(1),np.zeros((1,1)),np.full((1,1),.01),np.array([[[100.]],[[10.]]]),np.zeros(1))
originals = [value.copy() for value in positive_inputs]
index,costs,states = plan_candidates(*positive_inputs,action_penalty=.5)
assert index == 1 and np.allclose(costs,[5001.,50.01]) and np.allclose(states,[[[1.]],[[.1]]])
assert all(np.array_equal(value,before) for value,before in zip(positive_inputs,originals))
for overflowing in [
    lambda: plan_candidates(np.array([1e308]),np.array([[2.]]),np.zeros((1,1)),np.zeros((2,1,1)),np.zeros(1),1.),
    lambda: plan_candidates(np.zeros(1),np.zeros((1,1)),np.ones((1,1)),np.array([[[1e200]],[[1e199]]]),np.zeros(1)),
    lambda: plan_candidates(np.array([-1e308]),np.ones((1,1)),np.zeros((1,1)),np.zeros((2,1,1)),np.array([1e308])),
    lambda: plan_candidates(np.zeros(2),np.zeros((2,2)),np.full((2,1),1e154),np.ones((2,1,1)),np.zeros(2)),
    lambda: plan_candidates(np.zeros(1),np.zeros((1,1)),np.zeros((1,2)),np.full((2,1,2),1e154),np.zeros(1),1.),
    lambda: plan_candidates(np.zeros(1),np.zeros((1,1)),np.zeros((1,1)),np.full((2,1,1),1e100),np.zeros(1),1e200),
    lambda: plan_candidates(np.zeros(1),np.zeros((1,1)),np.ones((1,1)),np.full((2,1,1),1e154),np.zeros(1),1.),
]:
    try:
        overflowing()
    except FloatingPointError:
        pass
    else:
        raise AssertionError('Nonfinite active rollout, squared-distance, reduction, or objective intermediates must fail explicitly')
for dtype in (np.int16,np.uint8):
    initial = np.array([0],dtype=dtype)
    a,b = np.zeros((1,1),dtype=dtype),np.ones((1,1),dtype=dtype)
    actions = np.array([[[200]],[[10]]],dtype=dtype)
    goal = np.array([0],dtype=dtype)
    inputs = (initial,a,b,actions,goal)
    originals = [value.copy() for value in inputs]
    index,costs,states = plan_candidates(*inputs)
    assert index == 1 and np.array_equal(costs,[40000.,100.]), 'Integer action costs must select action 10 over 200'
    assert np.array_equal(states,[[[200.]],[[10.]]])
    assert costs.dtype.kind == 'f' and states.dtype.kind == 'f'
    _,goal_costs,_ = plan_candidates(initial,a,b,actions,np.array([200],dtype=dtype))
    assert np.array_equal(goal_costs,[0.,36100.]), 'Unsigned goal subtraction must not wrap'
    assert all(np.array_equal(value,before) for value,before in zip(inputs,originals))
    initial = np.array([200,200],dtype=dtype)
    a,b = np.full((2,2),200,dtype=dtype),np.full((2,2),200,dtype=dtype)
    actions = np.array([[[200,200],[200,200]],[[10,10],[10,10]]],dtype=dtype)
    goal = np.array([200,200],dtype=dtype)
    inputs = (initial,a,b,actions,goal)
    originals = [value.copy() for value in inputs]
    index,costs,states = plan_candidates(*inputs,action_penalty=2.)
    expected_states = [[[160000.,160000.],[64080000.,64080000.]],[[84000.,84000.],[33604000.,33604000.]]]
    expected_costs = [2*(159800**2+64079800**2)+2*4*200**2,2*(83800**2+33603800**2)+2*4*10**2]
    assert index == 1 and np.allclose(states,expected_states) and np.allclose(costs,expected_costs), 'Rollout matrix products and action penalties must not overflow integer dtypes'
    assert all(np.array_equal(value,before) for value,before in zip(inputs,originals))
for dtype in (np.int64,np.uint64):
    actions = np.array([[[10**9]]*24,[[5*10**8]]*24],dtype=dtype)
    index,costs,_ = plan_candidates(np.zeros(1,dtype=dtype),np.zeros((1,1),dtype=dtype),np.ones((1,1),dtype=dtype),actions,np.zeros(1,dtype=dtype),3.)
    assert index == 1 and np.allclose(costs,[9.6e19,2.4e19]), 'Cost reductions must not overflow signed or unsigned integer accumulators'
rng = np.random.default_rng(1)
a,b = rng.normal(size=(2,2)),rng.normal(size=(2,3));c=rng.normal(size=(4,3,3));z=np.array([1.,-1.]);goal=np.array([.5,.2])
_,costs,rollouts = plan_candidates(z,a,b,c,goal,.1)
for i in range(4):
    state=z.copy();cost=0
    for t in range(3):
        state=a@state+b@c[i,t];cost+=sum((state-goal)**2)+.1*sum(c[i,t]**2)
        assert np.allclose(state,rollouts[i,t])
    assert np.isclose(cost,costs[i])
print('World-model planning checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Learn action-conditioned dynamics, plan, and compare predictions to a simulator.

Known linear dynamics are the trusted simulator. Latents are observed states;
this is NOT an end-to-end pixel encoder or LeWorldModel/SIGReg reproduction.
"""
import json
import numpy as np


def world_experiment():
    rng=np.random.default_rng(18)
    true_a=np.array([[.9,.1],[0.,.95]]);true_b=np.array([[.5,0.],[.1,.4]])
    states=rng.normal(size=(300,2));actions=rng.uniform(-1,1,size=(300,2))
    next_states=states@true_a.T+actions@true_b.T
    noisy=next_states+rng.normal(size=next_states.shape)*.03
    design=np.column_stack([states,actions])
    fitted=np.linalg.lstsq(design[:240],noisy[:240],rcond=None)[0]
    learned_a,learned_b=fitted[:2].T,fitted[2:].T
    heldout=float(((design[240:]@fitted-next_states[240:])**2).mean())
    initial=np.array([-1.,-.5]);goal=np.array([1.,1.])
    candidates=rng.uniform(-1,1,size=(512,8,2))
    def costs(a,b,candidates,start):
        z=np.repeat(start[None,:],len(candidates),0);cost=np.zeros(len(candidates))
        for t in range(candidates.shape[1]):
            z=z@a.T+candidates[:,t]@b.T
            cost+=((z-goal)**2).sum(-1)+.02*(candidates[:,t]**2).sum(-1)
        return cost
    predicted=costs(learned_a,learned_b,candidates,initial)
    observed=costs(true_a,true_b,candidates,initial);choice=int(predicted.argmin())
    error_by_horizon=[]
    z_true,z_model=initial.copy(),initial.copy()
    for action in candidates[choice]:
        z_true=true_a@z_true+true_b@action;z_model=learned_a@z_model+learned_b@action
        error_by_horizon.append(float(((z_true-z_model)**2).mean()))
    # A biased model overestimates action effectiveness. Compare closed/open loop
    # using the same candidate count and planning calls per executed step.
    bad_b=learned_b*1.8
    state=initial.copy();closed=[]
    for _ in range(8):
        local=rng.uniform(-1,1,size=(128,4,2))
        chosen=int(costs(learned_a,bad_b,local,state).argmin())
        state=true_a@state+true_b@local[chosen,0];closed.append(state.tolist())
    # Conditional pixel-MSE mean for two distinct futures.
    futures=np.array([[1.,0.,0.],[0.,0.,1.]])
    return dict(seed=18,train_transitions=240,test_transitions=60,heldout_one_step_mse=heldout,
        learned_a=learned_a.tolist(),learned_b=learned_b.tolist(),selected_plan=choice,
        predicted_cost=float(predicted[choice]),simulator_cost=float(observed[choice]),
        multistep_mse=error_by_horizon,replanned_states=closed,ambiguous_pixel_mean=futures.mean(0).tolist(),
        limitation='Observed linear states, bounded sampled actions. Replanning uses extra observations and planning work; not a matched-cost superiority claim.')


if __name__=='__main__':
    print(json.dumps(world_experiment(),indent=2))
